# 02 · 两群体 SEIQR 基础模型

本 Notebook 用于：
1. 验证 ModelParams 参数配置
2. 运行 SEIQR ODE 求解
3. 计算 R₀（次代矩阵法）和群体免疫阈值
4. 可视化流行曲线与各仓室时序
5. 分析 β(t) 季节调制效果

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from model.params import ModelParams
from model.solver import solve_seiqr, extract_summary
from model.r0 import compute_R0, compute_herd_immunity_threshold
from plots._style import setup_style
from plots.epidemic_curve import plot_epidemic_curve, plot_seiqr_compartments, plot_beta_seasonal

setup_style()

from pathlib import Path
FIG_DIR = Path('../output/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. 参数配置

In [ ]:
# 从 config.yaml 加载（若不存在则使用默认值）
config_path = Path('../config.yaml')
if config_path.exists():
    p = ModelParams.from_yaml(config_path)
else:
    p = ModelParams()

print(p)
print(f'\n人口: 学生={p.N1:,}, 教职工={p.N2:,}, 合计={p.N:,}')
print(f'\n接触矩阵:')
print(p.contact_matrix())
print(f'\n初始状态向量:')
y0 = p.initial_state()
labels = ['S1','E1','I1','Q1','R1','S2','E2','I2','Q2','R2']
for l, v in zip(labels, y0):
    print(f'  {l} = {v:,.0f}')

## 2. R₀ 计算

In [ ]:
R0  = compute_R0(p)
HIT = compute_herd_immunity_threshold(R0)

print(f'基本再生数 R₀ = {R0:.4f}')
print(f'群体免疫阈值 HIT = {HIT:.2%} ({HIT*p.N:,.0f} 人需获得免疫)')
print(f'当前疫苗接种率: {p.vax_coverage:.2%} × 有效率 {p.vax_efficacy:.0%} = {p.vax_coverage*p.vax_efficacy:.4%}')
print(f'免疫缺口: {max(HIT - p.vax_coverage*p.vax_efficacy, 0):.2%}')

## 3. ODE 求解与流行曲线

In [ ]:
df = solve_seiqr(p)
summ = extract_summary(df, p)

print('模型关键指标:')
for k, v in summ.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

In [ ]:
# 流行曲线
fig = plot_epidemic_curve(
    df,
    title=f'H3N2 校园传播 SEIQR 模型（R₀={R0:.2f}）',
    output_path=FIG_DIR / '02_epidemic_curve.png',
    show=True
)

In [ ]:
# 各仓室时序图
fig = plot_seiqr_compartments(
    df, group='student',
    output_path=FIG_DIR / '02_compartments_student.png',
    show=True
)

In [ ]:
# β(t) 季节系数
fig = plot_beta_seasonal(
    df,
    output_path=FIG_DIR / '02_beta_seasonal.png',
    show=True
)

## 4. 参数敏感性快速检验

In [ ]:
from model.r0 import r0_sensitivity_table
from plots.sensitivity_plot import plot_r0_sensitivity_line

# β₀ 对 R₀ 的影响
beta_vals = np.linspace(0.05, 0.70, 50)
r0_tbl = r0_sensitivity_table(p, 'beta0', beta_vals)

fig = plot_r0_sensitivity_line(
    np.array(r0_tbl['values']), r0_tbl['R0'], r0_tbl['HIT'],
    param_label='基础传播系数 β₀',
    base_val=p.beta0,
    output_path=FIG_DIR / '02_r0_beta_sensitivity.png',
    show=True
)

print(f'当前 β₀={p.beta0:.3f} 对应 R₀={R0:.3f}')
print(f'使 R₀<1 需要 β₀ < {next((v for v,r in zip(r0_tbl["values"], r0_tbl["R0"]) if r < 1.0), "N/A"):.3f}')